# Historical Weekly Feature Panel

This notebook exports the invoice and calendar features from the Plan Book for all eligible historical SKU-weeks by market. Each row represents one country, SKU, segment, product type, and Monday. Candidate-price and synthetic inventory features remain in the separate scenario snapshots. Actual weekly sales are labels. Weeks without observed sales retain a missing label because listing and stock history are unavailable.

[PriceWise_Methodology_Plan.ipynb](../PriceWise_Methodology_Plan.ipynb)

## Set the Scope and Read the Invoices

Apply the project's US and CA, product type, segment, and positive-invoice filters. Keep invoice lines through August 23, 2026. Use order dates to group sales and require shipment strictly before Monday for feature inputs. Export each market separately to keep prices and category statistics market-specific.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import nbformat

project = Path.cwd()
if not (project/"valid_US.csv").is_file():
    project = project.parent
out = project/"feature"
as_of = pd.Timestamp("2026-08-23")
mondays = pd.date_range("2023-04-03", as_of + pd.Timedelta(days=1), freq="W-MON")
gap_start, gap_end = pd.Timestamp("2025-10-01"), pd.Timestamp("2026-01-01")
sku_key = ["product_id", "segment", "product_type"]
week_key = sku_key + ["week_start"]
family_key = ["segment", "product_family"]
types = ["Notebook", "Desktop", "Workstations"]
segments = ["Enterprise", "Personal"]
fields = ["country", *sku_key, "product_family", "order_date", "ship_date",
          "customer_id", "sales_channel", "state", "invoice_quantity",
          "invoice_revenue_country_currency", "month_fiscal", "quarter_fiscal"]
events = pd.read_csv(project/"pricewise_outputs/event_calendar.csv",
                     parse_dates=["event_date", "week_start"])
plan = nbformat.read(project/"PriceWise_Methodology_Plan.ipynb", as_version=4)
backlog = next(cell.source for cell in plan.cells if cell.source.startswith("### 3.2 Candidate features"))
names = {int(i):label.strip() for i,label in re.findall(r"^\| (\d+) \| ([^|]+) \| .+ \|$",backlog,re.M)}
assert set(names) == set(range(1,76))

def read_market(market):
    chunks = []
    for chunk in pd.read_csv(project/f"valid_{market}.csv", usecols=fields,
                             chunksize=250_000, dtype={"customer_id":"string", "state":"string"}):
        keep = (
            chunk.country.eq(market) & chunk.product_type.isin(types)
            & chunk.segment.isin(segments) & chunk.product_id.notna()
            & chunk.invoice_quantity.gt(0) & chunk.invoice_revenue_country_currency.gt(0)
        )
        rows = chunk.loc[keep].rename(columns={
            "invoice_quantity":"units", "invoice_revenue_country_currency":"sales"
        }).drop(columns="country").copy()
        for col in ("order_date", "ship_date"):
            rows[col] = pd.to_datetime(rows[col], format="%Y-%m-%d", errors="coerce")
        rows["product_family"] = rows.product_family.fillna("(Unlabeled Family)")
        chunks.append(rows.loc[rows.order_date.le(as_of)])
    return pd.concat(chunks, ignore_index=True)

## Calculate Weekly SKU and Family Features

Calculate features at each Monday from trailing visible invoices. Export active weeks from the first visible sale through the final cutoff. Retain and flag outcomes touching the coverage gap and mask each feature whose own lookback crosses it. Unit counts summarize recorded sales. Missing historical support stays empty. Family ranks use the eligible SKU-weeks with usable 28-day observations.

In [2]:
def divide(a, b):
    return a / b if b > 0 else np.nan

def date_arrays(g):
    return (
        g.order_date.to_numpy("datetime64[D]"),
        g.ship_date.to_numpy("datetime64[D]"),
        g.units.to_numpy(),
    )

def trailing(g):
    dates, ships, units = date_arrays(g)
    for ti, stamp in enumerate(mondays):
        t = stamp.to_datetime64().astype("datetime64[D]")
        left, right = np.searchsorted(dates, [t-np.timedelta64(56,"D"), t])
        visible = ships[left:right] < t
        ix = np.flatnonzero(visible) + left
        age = (t-dates[ix]).astype(int)
        yield ti, t, ix, age, units[ix]

def touches_gap(t, days):
    return t > gap_start and t-pd.Timedelta(days=days) < gap_end

def sparse_shares(codes, units, group_count):
    observed = codes >= 0
    if not observed.any():
        return np.zeros(group_count), False
    counts = np.bincount(codes[observed], weights=units[observed], minlength=group_count)
    return counts / counts.sum(), True

def sku_features(g):
    g = g.sort_values(["order_date","ship_date"], kind="stable")
    dates, ships, all_units = date_arrays(g)
    labels = g.product_family.to_numpy()
    observed = ships < np.datetime64(as_of.date()+pd.Timedelta(days=1))
    if not observed.any():
        return []
    # A sale is visible only after both its order and shipment dates.
    known = np.maximum(dates, ships)
    visibility_order = np.argsort(known)
    known = known[visibility_order]
    earliest = np.minimum.accumulate(dates[visibility_order])
    latest = np.maximum.accumulate(dates[visibility_order])
    customer, customers = pd.factorize(g.customer_id)
    channel, channels = pd.factorize(g.sales_channel)
    state, states = pd.factorize(g.state)
    direct_code = channels.get_loc("Direct") if "Direct" in channels else -1
    market_code = channels.get_loc("Marketplace") if "Marketplace" in channels else -1
    weights = 0.7 ** np.arange(4)
    records = []
    for ti, t, ix, age, units in trailing(g):
        visible_count = np.searchsorted(known, t)
        stamp = mondays[ti]
        if not visible_count:
            continue
        first = earliest[visible_count-1]
        last = latest[visible_count-1]
        current_family = labels[ix[-1]] if len(ix) else labels[
            visibility_order[visible_count-1]
        ]
        w = np.bincount((age-1)//7, weights=units, minlength=8)
        u7,u14,u28,u56 = [w[:i].sum() for i in (1,2,4,8)]
        r28, earlier = age<=28, age>28
        current, previous = ix[r28],ix[earlier]
        wins = units[r28]
        ch, has_ch = sparse_shares(channel[current],wins,len(channels))
        old_ch, has_old = sparse_shares(channel[previous],units[earlier],len(channels))
        cu, has_cu = sparse_shares(customer[current],wins,len(customers))
        st, has_st = sparse_shares(state[current],wins,len(states))
        direct = (ch[direct_code] if direct_code>=0 else 0.0) if has_ch else np.nan
        marketplace = (ch[market_code] if market_code>=0 else 0.0) if has_ch else np.nan
        earlier_direct = (old_ch[direct_code] if direct_code>=0 else 0.0) if has_old else np.nan
        returning = np.unique(customer[previous])
        returning = returning[returning>=0]
        daily = np.bincount(age[r28], weights=wins)
        by_feature = {
            1:u7,2:u14,3:u28,4:u56,5:np.log1p(u28),6:divide(u7*4,u28),
            7:u7/7-(u28-u7)/21,8:w[0]-w[1],9:np.dot(w[:4][::-1],[-.3,-.1,.1,.3]),
            10:np.average(w[:4],weights=weights),11:(w>0).sum(),12:int((t-last).astype(int)),
            13:len(np.unique(age[r28])),14:len(current),15:divide(u28,len(current)),
            16:divide((wins>1).sum(),len(current)),17:w.std(),18:divide(w.std(),w.mean()),
            19:divide(daily.max() if len(daily) else 0,u28),20:divide(u14,u56),
            29:int((t-first).astype(int)),31:direct,32:marketplace,33:(ch**2).sum() if has_ch else np.nan,
            34:(cu>0).sum(),35:cu.max() if has_cu else np.nan,36:cu[returning].sum() if has_cu else np.nan,
            37:(st**2).sum() if has_st else np.nan,38:direct-earlier_direct,
        }
        window_lengths = {1:7,2:14,3:28,4:56,5:28,6:28,7:28,8:14,9:28,10:28,11:56,
                          13:28,14:28,15:28,16:28,17:56,18:56,19:28,20:56,
                          31:28,32:28,33:28,34:28,35:28,36:56,37:28,38:56}
        for feature, days in window_lengths.items():
            if touches_gap(stamp,days):
                by_feature[feature] = np.nan
        if touches_gap(stamp, int((t-last).astype(int))):
            by_feature[12] = np.nan
        records.append({"week_start":stamp,"product_family":current_family,**by_feature})
    return records

def family_features(rows):
    summaries = []
    for group_key,g in rows.groupby(family_key):
        g = g.sort_values("order_date",kind="stable")
        points = []
        for ti,t,ix,age,units in trailing(g):
            points.append((mondays[ti],units[age<=7].sum(),units[age<=28].sum(),units.sum()))
        result = pd.DataFrame(points,columns=["week_start","family_7","family_28","family_56"])
        result[family_key[0]], result[family_key[1]] = group_key

        # At each cutoff, prior ISO-year totals still require visible shipments.
        dates,ships,units = date_arrays(g)
        years = g.order_date.dt.isocalendar().year.to_numpy(dtype=int)
        week_ids = g.order_date.dt.isocalendar().week.to_numpy(dtype=int)
        index_values = []
        for stamp in mondays:
            iso = stamp.isocalendar()
            t = stamp.to_datetime64().astype("datetime64[D]")
            visible = (years == iso.year-1) & (ships<t)
            if not visible.any():
                index_values.append(np.nan)
                continue
            # The known gap does not count toward the prior-year reference.
            keep = visible & ~((dates>=np.datetime64("2025-10-01")) & (dates<np.datetime64("2026-01-01")))
            totals = np.bincount(week_ids[keep],weights=units[keep],minlength=54)
            nonzero = totals[totals>0]
            index_values.append(totals[iso.week]/nonzero.mean() if len(nonzero) and totals[iso.week]>0 else np.nan)
        result[27] = index_values
        summaries.append(result)
    return pd.concat(summaries,ignore_index=True)

## Attach Actual Weekly Sales and Export Both Markets

Join the realized order-week quantities and local-currency revenue after constructing the features. Use the fiscal label recorded for Monday when available. Use an April-start calendar proxy for Mondays without a label and report the agreement between the proxy and observed transaction labels. Record whether the label may be used in backtests, along with feature coverage. Keep dates with no observed sale in the panel so missing labels remain visible.

In [3]:
def build_market(market, rows):
    sku_frames = []
    for group_key, g in rows.groupby(sku_key):
        points = sku_features(g)
        if points:
            sku_frames.append(pd.DataFrame(points).assign(**dict(zip(sku_key, group_key))))
    panel = pd.concat(sku_frames,ignore_index=True)
    panel = panel.merge(family_features(rows), on=family_key+["week_start"],
                        how="left",validate="many_to_one")
    dates = panel.week_start
    fiscal_year = dates.dt.year + dates.dt.month.ge(4).astype(int)
    fiscal_quarter = ((dates.dt.month-4)%12)//3+1
    panel[21] = dates.dt.month_name()
    panel[22] = "FY"+(fiscal_year%100).astype(str).str.zfill(2)+"Q"+fiscal_quarter.astype(str)
    source_dates = rows.order_date
    source_year = (source_dates.dt.year+source_dates.dt.month.ge(4).astype(int))%100
    source_quarter = ((source_dates.dt.month-4)%12)//3+1
    expected_fiscal = "FY"+source_year.astype(str).str.zfill(2)+"Q"+source_quarter.astype(str)
    fiscal_matches = (expected_fiscal == rows.quarter_fiscal).mean()
    observed_fiscal = rows.groupby("order_date").quarter_fiscal.agg(
        lambda values: values.mode().iat[0] if values.notna().any() else np.nan
    )
    observed_monday = dates.map(observed_fiscal)
    panel[22] = observed_monday.fillna(panel[22])
    print(f"{market} fiscal proxy agreement on invoice rows: {fiscal_matches:.2%}; "
          f"Monday labels from source: {observed_monday.notna().mean():.2%}")

    iso_week = dates.dt.isocalendar().week.astype(int)
    angle = 2*np.pi*(iso_week-1)/52.18
    panel[23],panel[24] = np.sin(angle),np.cos(angle)
    market_events = events[events.country.eq(market)]
    upcoming = np.sort(market_events.event_date.unique())
    positions = np.searchsorted(upcoming,dates.to_numpy())
    assert (positions<len(upcoming)).all()
    panel[25] = (pd.Series(upcoming[positions],index=panel.index)-dates).dt.days
    panel[26] = dates.isin(market_events.week_start).astype(int)
    for label in sorted(market_events.event.unique()):
        panel[f"event_{label}"] = dates.isin(
            market_events.loc[market_events.event.eq(label),"week_start"]
        )
    gap28 = dates.gt(gap_start) & (dates-pd.Timedelta(days=28)).lt(gap_end)
    gap56 = dates.gt(gap_start) & (dates-pd.Timedelta(days=56)).lt(gap_end)
    panel[28] = (panel.family_28/(panel.family_56-panel.family_28).replace(0,np.nan)-1).mask(gap56)
    panel[30] = (panel[3]/panel.family_28.replace(0,np.nan)).mask(gap28)
    panel[39] = panel.groupby(family_key+["week_start"])[3].rank(pct=True)
    panel[40] = (panel[6]-4*panel.family_7/panel.family_28.replace(0,np.nan)).mask(gap28)

    actual = rows.assign(week_start=source_dates-pd.to_timedelta(source_dates.dt.dayofweek,unit="D")).groupby(
        week_key, as_index=False
    ).agg(actual_units=("units","sum"),actual_revenue_local=("sales","sum"))
    actual["actual_asp_local"] = actual.actual_revenue_local/actual.actual_units
    panel = panel.merge(actual,on=week_key,how="left",validate="one_to_one")
    panel["outcome_week_complete"] = panel.week_start+pd.Timedelta(days=6)<=as_of
    panel["outcome_touches_coverage_gap"] = (
        panel.week_start.lt(gap_end) & (panel.week_start+pd.Timedelta(days=7)).gt(gap_start)
    )
    panel["lookback_56d_touches_coverage_gap"] = gap56.to_numpy()
    panel["eligible_for_sales_backtest"] = (
        panel.outcome_week_complete & ~panel.outcome_touches_coverage_gap
        & panel.actual_units.notna()
    )
    panel["country"] = market
    panel["currency"] = {"US":"USD","CA":"CAD"}[market]
    panel = panel.sort_values(week_key)
    assert not panel.duplicated(week_key).any()
    valid_windows = panel[[1,2,3,4]].dropna()
    assert (np.diff(valid_windows.to_numpy(),axis=1)>=0).all()
    assert not panel.loc[panel.eligible_for_sales_backtest,"actual_units"].isna().any()
    assert panel.loc[panel.lookback_56d_touches_coverage_gap,4].isna().all()
    id_cols=["country","currency"]+week_key+["product_family"]
    label_cols=["actual_units","actual_revenue_local","actual_asp_local",
                "outcome_week_complete","outcome_touches_coverage_gap",
                "lookback_56d_touches_coverage_gap","eligible_for_sales_backtest"]
    extras=[c for c in panel if isinstance(c,str) and c.startswith("event_")]
    return panel[id_cols+list(range(1,41))+extras+label_cols].rename(columns=names)

summaries, missing_rows = [], []
for market in ("US","CA"):
    rows = read_market(market)
    table = build_market(market,rows)
    table.to_csv(out/f"historical_weekly_features_{market}.csv",index=False)
    summary = {
        "market":market,"rows":len(table),"weeks":table.week_start.nunique(),
        "skus":table.product_id.nunique(),"feature_columns":40,
        "first_week":table.week_start.min().date(),"last_week":table.week_start.max().date(),
        "observed_backtest_labels":int(table.eligible_for_sales_backtest.sum())
    }
    summaries.append(summary)
    missing_rows.extend({"market":market,"feature":names[i],"missing":int(table[names[i]].isna().sum())}
                        for i in range(1,41))
    print(summary,flush=True)
pd.DataFrame(summaries).to_csv(out/"historical_weekly_feature_summary.csv",index=False)
pd.DataFrame(missing_rows).to_csv(out/"historical_weekly_feature_coverage.csv",index=False)
display(pd.DataFrame(summaries))

US fiscal proxy agreement on invoice rows: 97.41%; Monday labels from source: 99.13%
{'market': 'US', 'rows': 345399, 'weeks': 178, 'skus': 3015, 'feature_columns': 40, 'first_week': datetime.date(2023, 4, 3), 'last_week': datetime.date(2026, 8, 24), 'observed_backtest_labels': 65808}
CA fiscal proxy agreement on invoice rows: 97.13%; Monday labels from source: 99.11%
{'market': 'CA', 'rows': 288150, 'weeks': 178, 'skus': 2553, 'feature_columns': 40, 'first_week': datetime.date(2023, 4, 3), 'last_week': datetime.date(2026, 8, 24), 'observed_backtest_labels': 42124}
market   rows  weeks  skus  feature_columns first_week  last_week  observed_backtest_labels
    US 345399    178  3015               40 2023-04-03 2026-08-24                     65808
    CA 288150    178  2553               40 2023-04-03 2026-08-24                     42124
